# Sanity Check - Step 05: Downsampling

Überprüft:
- Downsampling zu 512 Hz erfolgreich
- Datenlänge reduziert wie erwartet
- Signalqualität erhalten
- Vergleich Vorher (2048 Hz?) vs. Nachher (512 Hz)

In [ ]:
import sys
import mne
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(str(Path.cwd().parent / 'eeg_pipeline'))
import config

print("Setup erfolgreich")

In [ ]:
subject_id = config.SUBJECTS[0]

# Step 04 Output laden (VORHER)
p1_path_before = config.OUTPUT_DIR / f"sub-{subject_id}_P1_ica_cleaned.fif"
p2_path_before = config.OUTPUT_DIR / f"sub-{subject_id}_P2_ica_cleaned.fif"

raw_p1_before = mne.io.read_raw_fif(str(p1_path_before), preload=False)
raw_p2_before = mne.io.read_raw_fif(str(p2_path_before), preload=False)

print(f"\n=== VORHER (Step 04 Output - ICA Cleaned) ===\n")
print(f"Person 1:")
print(f"  Sampling Rate: {raw_p1_before.info['sfreq']} Hz")
print(f"  Datenlänge: {raw_p1_before.n_times} Samples")
print(f"  Dauer: {raw_p1_before.times[-1]:.2f} Sekunden")
print(f"  Geschätzte Dateigröße: {raw_p1_before.n_times * len(raw_p1_before.ch_names) * 8 / 1e6:.2f} MB")

print(f"\nPerson 2:")
print(f"  Sampling Rate: {raw_p2_before.info['sfreq']} Hz")
print(f"  Datenlänge: {raw_p2_before.n_times} Samples")
print(f"  Dauer: {raw_p2_before.times[-1]:.2f} Sekunden")
print(f"  Geschätzte Dateigröße: {raw_p2_before.n_times * len(raw_p2_before.ch_names) * 8 / 1e6:.2f} MB")

In [ ]:
# Step 05 Output laden (NACHHER)
p1_path_after = config.OUTPUT_DIR / f"sub-{subject_id}_P1_downsampled.fif"
p2_path_after = config.OUTPUT_DIR / f"sub-{subject_id}_P2_downsampled.fif"

raw_p1_after = mne.io.read_raw_fif(str(p1_path_after), preload=False)
raw_p2_after = mne.io.read_raw_fif(str(p2_path_after), preload=False)

print(f"\n=== NACHHER (Step 05 Output - Downsampled) ===\n")
print(f"Person 1:")
print(f"  Sampling Rate: {raw_p1_after.info['sfreq']} Hz")
print(f"  Datenlänge: {raw_p1_after.n_times} Samples")
print(f"  Dauer: {raw_p1_after.times[-1]:.2f} Sekunden")
print(f"  Geschätzte Dateigröße: {raw_p1_after.n_times * len(raw_p1_after.ch_names) * 8 / 1e6:.2f} MB")

print(f"\nPerson 2:")
print(f"  Sampling Rate: {raw_p2_after.info['sfreq']} Hz")
print(f"  Datenlänge: {raw_p2_after.n_times} Samples")
print(f"  Dauer: {raw_p2_after.times[-1]:.2f} Sekunden")
print(f"  Geschätzte Dateigröße: {raw_p2_after.n_times * len(raw_p2_after.ch_names) * 8 / 1e6:.2f} MB")

In [ ]:
print(f"\n=== DOWNSAMPLING ANALYSE ===\n")

downsample_ratio_p1 = raw_p1_before.info['sfreq'] / raw_p1_after.info['sfreq']
downsample_ratio_p2 = raw_p2_before.info['sfreq'] / raw_p2_after.info['sfreq']

print(f"Person 1:")
print(f"  Sampling Rate Änderung: {raw_p1_before.info['sfreq']} Hz → {raw_p1_after.info['sfreq']} Hz")
print(f"  Downsampling Faktor: {downsample_ratio_p1:.1f}x")
print(f"  Samples Reduktion: {raw_p1_before.n_times} → {raw_p1_after.n_times}")
print(f"  Samples reduziert um: {(1 - raw_p1_after.n_times/raw_p1_before.n_times)*100:.1f}%")
print(f"  Erwartete Datengröße Reduktion: {(1 - 1/downsample_ratio_p1)*100:.1f}%")

print(f"\nPerson 2:")
print(f"  Sampling Rate Änderung: {raw_p2_before.info['sfreq']} Hz → {raw_p2_after.info['sfreq']} Hz")
print(f"  Downsampling Faktor: {downsample_ratio_p2:.1f}x")
print(f"  Samples Reduktion: {raw_p2_before.n_times} → {raw_p2_after.n_times}")
print(f"  Samples reduziert um: {(1 - raw_p2_after.n_times/raw_p2_before.n_times)*100:.1f}%")
print(f"  Erwartete Datengröße Reduktion: {(1 - 1/downsample_ratio_p2)*100:.1f}%")

In [ ]:
# Lade Daten für Signalvergleich
raw_p1_before.load_data()
raw_p1_after.load_data()

# Wähle einen EEG Kanal
eeg_picks = mne.pick_types(raw_p1_before.info, eeg=True)
ch_idx = 15
ch_name = mne.pick_types(raw_p1_before.info, eeg=True, ret_names=True)[0][ch_idx]

# Zeitfenster: erste 5 Sekunden
t_end = min(5, raw_p1_before.times[-1])

# Für VORHER: alle Samples in diesem Fenster
t_idx_end_before = int(t_end * raw_p1_before.info['sfreq'])
data_before = raw_p1_before.get_data(picks=[eeg_picks[ch_idx]])[0, :t_idx_end_before]
times_before = raw_p1_before.times[:t_idx_end_before]

# Für NACHHER: Samples im gleichen Zeitfenster (aber weniger)
t_idx_end_after = int(t_end * raw_p1_after.info['sfreq'])
data_after = raw_p1_after.get_data(picks=[eeg_picks[ch_idx]])[0, :t_idx_end_after]
times_after = raw_p1_after.times[:t_idx_end_after]

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# VORHER
ax1 = axes[0]
ax1.plot(times_before, data_before, 'b-', linewidth=0.5, label=f'Original ({raw_p1_before.info["sfreq"]:.0f} Hz)')
ax1.set_ylabel('Amplitude (µV)')
ax1.set_title(f'VORHER: Kanal {ch_name} - Original Sampling Rate ({raw_p1_before.info["sfreq"]:.0f} Hz)')
ax1.grid(True, alpha=0.3)
ax1.legend()

# NACHHER
ax2 = axes[1]
ax2.plot(times_after, data_after, 'g-', linewidth=0.5, label=f'Downsampled ({raw_p1_after.info["sfreq"]:.0f} Hz)', marker='o', markersize=2, alpha=0.7)
ax2.set_xlabel('Zeit (Sekunden)')
ax2.set_ylabel('Amplitude (µV)')
ax2.set_title(f'NACHHER: Kanal {ch_name} - Downsampled ({raw_p1_after.info["sfreq"]:.0f} Hz) - erste {t_end:.1f}s')
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

print(f"\nVergleichter Kanal: {ch_name}")
print(f"Zeitfenster: {t_end:.1f} Sekunden")
print(f"VORHER: {len(data_before)} Samples in {t_end}s ({raw_p1_before.info['sfreq']:.0f} Hz)")
print(f"NACHHER: {len(data_after)} Samples in {t_end}s ({raw_p1_after.info['sfreq']:.0f} Hz)")

In [ ]:
# PSD Vergleich - zeigt ob die Abtastung die Signaleigenschaften bewahrt
fig = plt.figure(figsize=(14, 6))

# VORHER
plt.subplot(1, 2, 1)
raw_p1_before_eeg = raw_p1_before.copy().pick_types(eeg=True)
raw_p1_before_eeg.plot_psd(fmax=200, ax=plt.gca(), show=False)
plt.axvline(x=raw_p1_after.info['sfreq']/2, color='red', linestyle='--', label=f'Nyquist ({raw_p1_after.info["sfreq"]/2:.0f} Hz)')
plt.title('VORHER: Power Spectral Density')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Power (µV²/Hz)')
plt.legend()

# NACHHER
plt.subplot(1, 2, 2)
raw_p1_after_eeg = raw_p1_after.copy().pick_types(eeg=True)
raw_p1_after_eeg.plot_psd(fmax=256, ax=plt.gca(), show=False)  # Nyquist ist 256 Hz bei 512 Hz sampling
plt.axvline(x=raw_p1_after.info['sfreq']/2, color='red', linestyle='--', label=f'Nyquist ({raw_p1_after.info["sfreq"]/2:.0f} Hz)')
plt.title(f'NACHHER: Power Spectral Density (downsampled zu {raw_p1_after.info["sfreq"]:.0f} Hz)')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Power (µV²/Hz)')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Amplituden Statistiken
eeg_picks = mne.pick_types(raw_p1_before.info, eeg=True)
data_before_all = raw_p1_before.get_data(picks=eeg_picks)
data_after_all = raw_p1_after.get_data(picks=eeg_picks)

print(f"\n=== AMPLITUDEN STATISTIKEN (Person 1) ===\n")

print(f"VORHER ({raw_p1_before.info['sfreq']:.0f} Hz):")
print(f"  Min: {np.min(data_before_all):.6f} µV")
print(f"  Max: {np.max(data_before_all):.6f} µV")
print(f"  Mean: {np.mean(data_before_all):.6f} µV")
print(f"  Std: {np.std(data_before_all):.6f} µV")

print(f"\nNACHHER ({raw_p1_after.info['sfreq']:.0f} Hz):")
print(f"  Min: {np.min(data_after_all):.6f} µV")
print(f"  Max: {np.max(data_after_all):.6f} µV")
print(f"  Mean: {np.mean(data_after_all):.6f} µV")
print(f"  Std: {np.std(data_after_all):.6f} µV")

print(f"\nÄNDERUNG:")
print(f"  Std-Änderung: {(np.std(data_after_all) - np.std(data_before_all)):.6f} µV")
print(f"  Std % Änderung: {((np.std(data_after_all) - np.std(data_before_all)) / np.std(data_before_all) * 100):.2f}%")

In [ ]:
print(f"\n=== SANITY CHECK ZUSAMMENFASSUNG ===\n")

checks = []

# 1. Sampling Rate sollte zu config.DOWNSAMPLE_SFREQ reduziert sein
correct_sfreq_p1 = raw_p1_after.info['sfreq'] == config.DOWNSAMPLE_SFREQ
correct_sfreq_p2 = raw_p2_after.info['sfreq'] == config.DOWNSAMPLE_SFREQ
checks.append((f"Sampling Rate zu {config.DOWNSAMPLE_SFREQ} Hz reduziert (P1)", correct_sfreq_p1))
checks.append((f"Sampling Rate zu {config.DOWNSAMPLE_SFREQ} Hz reduziert (P2)", correct_sfreq_p2))

# 2. Anzahl Samples sollte um Downsampling Faktor reduziert sein
expected_samples_p1 = int(raw_p1_before.n_times * config.DOWNSAMPLE_SFREQ / raw_p1_before.info['sfreq'])
samples_correct = abs(raw_p1_after.n_times - expected_samples_p1) <= 1  # 1 Sample Toleranz
checks.append(("Sample-Anzahl korrekt reduziert", samples_correct))

# 3. Zeitdauer sollte gleich sein
same_duration = abs(raw_p1_before.times[-1] - raw_p1_after.times[-1]) < 0.01
checks.append(("Zeitdauer gleich", same_duration))

# 4. Kanal-Namen sollten gleich sein
same_channels = raw_p1_before.ch_names == raw_p1_after.ch_names
checks.append(("Kanal-Namen gleich", same_channels))

# 5. Signalstärke sollte ähnlich sein (nicht zu anders)
std_diff_ratio = abs(np.std(data_after_all) - np.std(data_before_all)) / np.std(data_before_all)
signal_preserved = std_diff_ratio < 0.1  # Weniger als 10% Änderung
checks.append(("Signal-Qualität erhalten (<10% Std Änderung)", signal_preserved))

# 6. Keine NaN oder Inf
no_nan_inf = not (np.isnan(data_after_all).any() or np.isinf(data_after_all).any())
checks.append(("Keine NaN/Inf Werte", no_nan_inf))

# 7. Dateigrößre sollte kleiner sein
size_before = raw_p1_before.n_times * len(raw_p1_before.ch_names) * 8
size_after = raw_p1_after.n_times * len(raw_p1_after.ch_names) * 8
size_reduced = size_after < size_before
checks.append(("Dateigröße reduziert", size_reduced))

for check_name, result in checks:
    status = "✓ PASS" if result else "✗ FAIL"
    print(f"{status}: {check_name}")

all_pass = all(result for _, result in checks)
print(f"\n{'='*50}")
if all_pass:
    print("✓ ALLE CHECKS BESTANDEN")
else:
    print("✗ EINIGE CHECKS FEHLGESCHLAGEN")
print(f"{'='*50}")